<img src="http://imgur.com/1ZcRyrc.png" style="float: left; margin: 20px; height: 55px">

# Using Pre-trained Models in Keras

_Author: Tim Book_

In [1]:
import os
import shutil

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tensorflow import keras
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.layers import RandomFlip, RandomRotation, RandomZoom
from tensorflow.keras.applications.vgg16 import VGG16
from tensorflow.keras.utils import image_dataset_from_directory

In [2]:
train_labels = pd.read_csv('butterflies/Training_set.csv')
C = train_labels.label.nunique()

Create datasets as in the previous section.

In [3]:
train_ds = image_dataset_from_directory(
    'data/train',
    batch_size=32,
    subset='training',
    seed=123,
    validation_split=0.2
).map(lambda x, y: (x/255, y))

val_ds = image_dataset_from_directory(
    'data/train',
    batch_size=32,
    subset='validation',
    seed=123,
    validation_split=0.2
).map(lambda x, y: (x/255, y))

Found 6499 files belonging to 75 classes.
Using 5200 files for training.
Found 6499 files belonging to 75 classes.
Using 1299 files for validation.


In [4]:
conv_base = VGG16(
    weights='imagenet',
    include_top=False
)

for layer in conv_base.layers:
    layer.trainable = False

In [5]:
conv_base.summary()

Model: "vgg16"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
input_1 (InputLayer)         [(None, None, None, 3)]   0         
_________________________________________________________________
block1_conv1 (Conv2D)        (None, None, None, 64)    1792      
_________________________________________________________________
block1_conv2 (Conv2D)        (None, None, None, 64)    36928     
_________________________________________________________________
block1_pool (MaxPooling2D)   (None, None, None, 64)    0         
_________________________________________________________________
block2_conv1 (Conv2D)        (None, None, None, 128)   73856     
_________________________________________________________________
block2_conv2 (Conv2D)        (None, None, None, 128)   147584    
_________________________________________________________________
block2_pool (MaxPooling2D)   (None, None, None, 128)   0     

In [6]:
data_aug = keras.models.Sequential([
    RandomFlip('horizontal_and_vertical', input_shape=(256, 256, 3)),
    RandomRotation(0.2),
    RandomZoom(height_factor=(-0.2, 0.2), width_factor=(-0.2, 0.2)),
])

In [7]:
inputs = keras.Input(shape=(256, 256, 3))
x = data_aug(inputs)
# Only necessary since we did a data augmentation step!
x = keras.applications.vgg16.preprocess_input(x)
x = conv_base(x)
x = Flatten()(x)
x = Dense(16, activation='relu')(x)
x = Dropout(0.5)(x)
outputs = Dense(C, activation='softmax')(x)

model = keras.Model(inputs, outputs)

In [8]:
model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='rmsprop',
    metrics=['acc']
)

In [9]:
model.summary()

Model: "model"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
input_2 (InputLayer)         [(None, 256, 256, 3)]     0         
_________________________________________________________________
sequential (Sequential)      (None, 256, 256, 3)       0         
_________________________________________________________________
tf.__operators__.getitem (Sl (None, 256, 256, 3)       0         
_________________________________________________________________
tf.nn.bias_add (TFOpLambda)  (None, 256, 256, 3)       0         
_________________________________________________________________
vgg16 (Functional)           (None, None, None, 512)   14714688  
_________________________________________________________________
flatten (Flatten)            (None, 32768)             0         
_________________________________________________________________
dense (Dense)                (None, 16)                524304

In [10]:
model.fit(train_ds, validation_data=val_ds, epochs=1)

163/163 [==============================] - 922s 6s/step - loss: 4.3348 - acc: 0.0179 - val_loss: 4.3175 - val_acc: 0.0231
